In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from utils import safe_open_json

graph_name = "amazon"
split = "test"


In [2]:
def load_metrics_df(json_files, max_agents=None, max_steps=None):

    data = []
    for json_file in json_files:
        log_data = safe_open_json(json_file)

        if log_data is None:
            continue  # Skip corrupted files

        # Extract key information from each log entry
        record = {
            "question_id": str(json_file).split("/")[-1].replace(".json", ""),
            "question": log_data.get("question", ""),
            "answer_indices": log_data.get("answer_indices", []),
            "trajectories": log_data.get("trajectories", []),
            "time_taken": log_data.get("time_taken", None),
        }

        if max_agents is None:
            max_agents = len(record["trajectories"])

        if len(record["trajectories"]) < max_agents:
            print(
                f"Warning: Expected {max_agents} agents but found {len(record['trajectories'])} in {json_file}"
            )
            max_agents = len(record["trajectories"])

        trajectories = record["trajectories"][:max_agents]

        latency = max([traj["step_times"][min(max_steps - 1, len(traj["step_times"]) - 1)] for traj in trajectories])

        agents_answer_indices = [
            traj.get("agent_answer_indices", [])
            for traj in trajectories
            if traj.get("steps") <= max_steps
        ]
        flat = [idx for sublist in agents_answer_indices for idx in sublist]
        counts = Counter(flat)
        first_seen = {}
        for i, idx in enumerate(flat):
            first_seen.setdefault(idx, i)

        record["latency"] = latency
        record["combined_answer_indices"] = sorted(
            counts.keys(), key=lambda x: (-counts[x], first_seen[x])
        )

        data.append(record)

    df = pd.DataFrame(data).reset_index(drop=True)

    df["recall@all"] = df.apply(
        lambda row: len(
            set(row["answer_indices"]).intersection(set(row["combined_answer_indices"]))
        )
        / len(set(row["answer_indices"])),
        axis=1,
    )
    df["hit@1"] = df.apply(
        lambda row: (
            row["combined_answer_indices"][0] in row["answer_indices"]
            if row["combined_answer_indices"]
            else False
        ),
        axis=1,
    )
    df["hit@5"] = df.apply(
        lambda row: len(
            set(row["answer_indices"]).intersection(set(row["combined_answer_indices"][:5]))
        )
        > 0,
        axis=1,
    )
    df["hit@10"] = df.apply(
        lambda row: len(
            set(row["answer_indices"]).intersection(set(row["combined_answer_indices"][:10]))
        )
        > 0,
        axis=1,
    )
    df["recall@10"] = df.apply(
        lambda row: len(
            set(row["answer_indices"]).intersection(set(row["combined_answer_indices"][:10]))
        )
        / len(set(row["answer_indices"])),
        axis=1,
    )
    df["recall@20"] = df.apply(
        lambda row: len(
            set(row["answer_indices"]).intersection(set(row["combined_answer_indices"][:20]))
        )
        / len(set(row["answer_indices"])),
        axis=1,
    )

    def reciprocal_rank(row):
        for rank, idx in enumerate(row["combined_answer_indices"], 1):
            if idx in row["answer_indices"]:
                return 1 / rank
        return 0

    df["MRR"] = df.apply(reciprocal_rank, axis=1)

    return df

In [3]:
from pathlib import Path
import pandas as pd

tools_to_remove = [None]
tools_to_remove.extend(
    [
        # "find_paths",
        "neighbor_queries",
        "neighbor_filters",
        "search_in_neighborhood",
    ]
)
model_name = "graph_explorer_gpt-4.1"


def load_df_for(tool_to_remove):
    if tool_to_remove is None:
        logs_dir = Path(f"../data/experiments/{graph_name}/{model_name}/{split}")
    else:
        logs_dir = Path(
            f"../data/ablations/{graph_name}/{model_name}_without_{tool_to_remove}/{split}"
        )

    json_files = sorted(logs_dir.glob("*.json"), key=lambda f: f.stat().st_ctime)
    df = load_metrics_df(json_files, max_agents=3, max_steps=15)

    if "question_id" not in df.columns:
        raise ValueError(f"`question_id` not found in df loaded from {logs_dir}")

    # Safety: if somehow you have multiple rows per question_id, keep one deterministically
    if df["question_id"].duplicated().any():
        df = df.sort_values("question_id").drop_duplicates("question_id", keep="last")

    return df, logs_dir


def summarize(df):
    return [
        ("n", len(df)),
        ("Hit@1", float(round(df["hit@1"].mean(), 3))),
        ("Hit@5", float(round(df["hit@5"].mean(), 3))),
        ("Recall@10", float(round(df["recall@10"].mean(), 3))),
        ("Recall@20", float(round(df["recall@20"].mean(), 3))),
        ("MRR", float(round(df["MRR"].mean(), 3))),
        ("Latency", float(round(df["latency"].mean(), 3))),
    ]


# 1) Load all dfs first
dfs = {}
dirs = {}
for t in tools_to_remove:
    try:
        df, d = load_df_for(t)
        dfs[t] = df
        dirs[t] = d
    except Exception as e:
        print(f"Error loading {t} from {d if 'd' in locals() else 'unknown dir'}: {e}")

# 2) Compute shared question_id set (inner join key)
if not dfs:
    raise RuntimeError("No dataframes loaded; cannot compare.")

shared_ids = set.intersection(*[set(df["question_id"]) for df in dfs.values()])
shared_ids = sorted(shared_ids)

# # Optional: cap to 400 *after* intersection (reproducible)
# shared_ids = shared_ids[:400]

# 3) Filter each df to the same questions and print metrics
for t in tools_to_remove:
    if t not in dfs:
        continue

    df = dfs[t]
    df_shared = df[df["question_id"].isin(shared_ids)].copy()

    print(f"Results for {model_name} without {t}:")
    print(f"  shared_n: {len(df_shared)} (of {len(df)} originally)")
    for k, v in summarize(df_shared):
        print(f"  {k}: {v}")
    print()

Results for graph_explorer_gpt-4.1 without None:
  shared_n: 200 (of 1632 originally)
  n: 200
  Hit@1: 0.585
  Hit@5: 0.755
  Recall@10: 0.51
  Recall@20: 0.602
  MRR: 0.662
  Latency: 10.014

Results for graph_explorer_gpt-4.1 without neighbor_queries:
  shared_n: 200 (of 200 originally)
  n: 200
  Hit@1: 0.56
  Hit@5: 0.715
  Recall@10: 0.488
  Recall@20: 0.579
  MRR: 0.635
  Latency: 9.887

Results for graph_explorer_gpt-4.1 without neighbor_filters:
  shared_n: 200 (of 200 originally)
  n: 200
  Hit@1: 0.555
  Hit@5: 0.73
  Recall@10: 0.507
  Recall@20: 0.599
  MRR: 0.641
  Latency: 10.803

Results for graph_explorer_gpt-4.1 without search_in_neighborhood:
  shared_n: 200 (of 200 originally)
  n: 200
  Hit@1: 0.545
  Hit@5: 0.685
  Recall@10: 0.47
  Recall@20: 0.554
  MRR: 0.613
  Latency: 10.669

